# LMI — ذكاء سوق العمل اليمني · تشغيل كامل من Google Drive
**قبل البدء:** Runtime → Change runtime type → **T4 GPU**.

**مرة واحدة فقط:** ارفع `LMI_PROJECT.zip` إلى جذر **My Drive** (سحب وإفلات من المتصفح).

ما يفعله الدفتر:
1. يربط Drive ويفك المشروع إلى `My Drive/LMI_PROJECT` (مرة واحدة).
2. يدرّب نماذج AraBERT الثلاثة **ويحفظها في Drive مباشرة** — ويتخطى أي نموذج مدرب مسبقاً (إن انقطعت الجلسة أعد التشغيل فيكمل).
3. يقيّم على test ويولّد `docs/RESULTS.md`.
4. يشغّل الاختبارات ثم **التطبيق مربوطاً بالنماذج** برابط عام.

الأجزاء المنفذة مسبقاً على البيانات الكاملة (التدقيق، خطوط الأساس، LSTM) موجودة في الحزمة ولا تُعاد.
**تحديث AMP:** أصبح تصحيح تحذير scheduler ضمن الدفتر قبل بدء التدريب. التحذير وحده ليس فشل تدريب، ونتائج validation لا تساوي نتائج test. إذا اكتملت مهمة مسبقاً ويحتوي مجلدها على `training_config.json` فسيتخطاها الدفتر؛ الإصلاح يؤثر على عمليات التدريب الجديدة فقط.


## 1) ربط Drive وتجهيز المشروع

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, shutil, subprocess, json, sys, time
MY = "/content/drive/MyDrive"
PROJ = f"{MY}/LMI_PROJECT"                       # المشروع الدائم في Drive (النماذج والتقارير تُحفظ هنا)
if not os.path.exists(f"{PROJ}/src/common.py"):
    z = f"{MY}/LMI_PROJECT.zip"
    assert os.path.exists(z), "ارفع LMI_PROJECT.zip إلى جذر My Drive أولاً"
    print("فك الضغط إلى Drive…"); zipfile.ZipFile(z).extractall(MY)
# نسخة عمل محلية سريعة للكود والبيانات (القراءة من Drive بطيئة أثناء التدريب)
ROOT = "/content/lmi"
if not os.path.exists(f"{ROOT}/src/common.py"):
    shutil.copytree(PROJ, ROOT, ignore=shutil.ignore_patterns("models_dl", "reports"))
os.chdir(ROOT); sys.path.insert(0, ROOT)
os.environ["LMI_FINAL_DIR"]   = f"{ROOT}/FINAL_TRAINING_DATASET"
os.environ["LMI_MODELS_DIR"]  = f"{PROJ}/models_dl"     # ← النماذج في Drive
os.environ["LMI_REPORTS_DIR"] = f"{PROJ}/reports"       # ← التقارير في Drive
print("المشروع:", PROJ); print(sorted(os.listdir(PROJ)))

In [ ]:
!pip -q install "transformers>=4.44" seqeval sklearn-crfsuite arabert accelerate safetensors streamlit plotly
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠ لا يوجد GPU — التدريب سيكون بطيئاً جداً")

### إصلاح فعلي لتحذير AMP / Scheduler (تلقائي)
هذه الخلية تُنشئ نسخاً احتياطية من سكربتات تدريب التصنيف وNER، ثم تمنع تحديث معدل التعلّم عند تخطي خطوة optimizer بسبب AMP. تعالج المشكلة في نسخة `/content/lmi` وفي المشروع الدائم على Drive. لا تعدّل الأوزان أو البيانات، ولا تعيد تدريب نموذج اكتمل سابقاً.


In [ ]:
# AMP scheduler fix: applies to local training code and the persistent Drive source.
#!/usr/bin/env python3
"""Patch incorrect LR scheduler stepping when AMP skips optimizer updates.

Safe/idempotent: backs up changed Python modules, validates syntax before writing.
Both a fast local Colab worktree and a persistent Drive tree may be patched.
This does NOT alter existing weights or claim to retrain models.
"""
from __future__ import annotations

import argparse
import ast
import datetime as dt
import os
import re
from pathlib import Path

MARKER = "AMP_SAFE_SCHEDULER_STEP_V1"
FILES = ("src/training/train_classifier.py", "src/training/train_ner.py")
PATTERN = re.compile(
    r"^(?P<indent>[ \t]*)scaler\.step\(opt\)\r?\n"
    r"(?P=indent)scaler\.update\(\)\r?\n"
    r"(?P=indent)opt\.zero_grad\((?:set_to_none=True)?\)\r?\n"
    r"(?P=indent)sch\.step\(\)", re.MULTILINE,
)


def patch_text(source: str, name: str) -> tuple[str, bool]:
    if MARKER in source:
        return source, False
    matches = list(PATTERN.finditer(source))
    if len(matches) != 1:
        raise RuntimeError(f"{name}: expected exactly one optimizer/scheduler block, found {len(matches)}; not changing file")

    indent = matches[0].group("indent")
    replacement = "\n".join((
        f"{indent}# {MARKER}: schedule only after a real optimizer update",
        f"{indent}scale_before_update = scaler.get_scale()",
        f"{indent}scaler.step(opt)",
        f"{indent}scaler.update()",
        f"{indent}step_skipped_by_amp = bool(amp and scaler.get_scale() < scale_before_update)",
        f"{indent}opt.zero_grad(set_to_none=True)",
        f"{indent}if not step_skipped_by_amp:",
        f"{indent}    optimizer_updates += 1",
        f"{indent}    sch.step()",
        f"{indent}else:",
        f"{indent}    amp_skipped_updates += 1",
    ))
    source = source[:matches[0].start()] + replacement + source[matches[0].end():]

    # Counters reset once per epoch, before the first minibatch is processed.
    epoch_counter = re.compile(r"^(?P<indent>[ \t]*)tot, n = 0\.0, 0[ \t]*$", re.MULTILINE)
    found = list(epoch_counter.finditer(source))
    if len(found) != 1:
        raise RuntimeError(f"{name}: no unique epoch counter initialization; not changing file")
    m = found[0]
    new_counter = m.group(0) + "\n" + m.group("indent") + "optimizer_updates = 0\n" + m.group("indent") + "amp_skipped_updates = 0"
    source = source[:m.start()] + new_counter + source[m.end():]

    # Save AMP diagnostics alongside validation metrics in training_history.csv.
    target = "        hist.append(row)"
    if source.count(target) != 1:
        raise RuntimeError(f"{name}: could not add AMP diagnostics to history")
    source = source.replace(target,
        "        row['optimizer_updates'] = optimizer_updates\n"
        "        row['amp_skipped_updates'] = amp_skipped_updates\n"
        "        hist.append(row)", 1)

    # Do not silently accept an epoch in which no weights could be updated.
    if name == "train_classifier.py":
        entry = "        lg = predict(model, dl_va, dev, amp)"
    else:
        entry = "        model.eval()"
    if source.count(entry) != 1:
        raise RuntimeError(f"{name}: could not insert update-validation guard")
    source = source.replace(entry,
        "        if optimizer_updates == 0:\n"
        "            raise RuntimeError('AMP skipped every optimizer step in this epoch; investigate nonfinite gradients.')\n"
        "        if amp_skipped_updates:\n"
        "            print(f'[AMP] epoch {ep}: skipped {amp_skipped_updates} nonfinite-gradient update(s)', flush=True)\n"
        + entry, 1)
    ast.parse(source, filename=name)
    return source, True


def patch_project(project_root: Path) -> None:
    if not (project_root / "src" / "common.py").exists():
        raise FileNotFoundError(f"Project not found: {project_root}")
    prepared = []
    for rel in FILES:
        p = project_root / rel
        if not p.is_file():
            raise FileNotFoundError(f"Required training source missing: {p}")
        raw = p.read_text(encoding="utf-8")
        edited, changed = patch_text(raw, p.name)
        prepared.append((p, raw, edited, changed))
    # All files validated before any changes are committed.
    for p, original, edited, changed in prepared:
        if not changed:
            print("ALREADY_PATCHED:", p)
            continue
        suffix = dt.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        bak = p.with_name(p.name + ".before_amp_fix_" + suffix + ".bak")
        bak.write_text(original, encoding="utf-8")
        tmp = p.with_name(p.name + ".tmp_amp_fix")
        tmp.write_text(edited, encoding="utf-8")
        os.replace(tmp, p)
        print("PATCHED:", p)
        print("BACKUP:", bak)
    print("SUCCESS: AMP scheduler is guarded; Python files syntax-checked. Models and datasets untouched.")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--project-root", default="/content/lmi")
    ap.add_argument("--drive-project", help="Optionally patch persistent project source in Google Drive")
    args = ap.parse_args()
    roots = [Path(args.project_root)]
    if args.drive_project:
        roots.append(Path(args.drive_project))
    for root in dict.fromkeys(roots):
        patch_project(root)



patch_project(Path(ROOT))
patch_project(Path(PROJ))


## 2) تدريب AraBERT (يتخطى المدرب مسبقاً — آمن لإعادة التشغيل)
أفضل Checkpoint على validation فقط؛ test لا يُقرأ أثناء التدريب.

In [ ]:
M = os.environ["LMI_MODELS_DIR"]
def trained(d): return os.path.exists(f"{d}/config.json") and os.path.exists(f"{d}/training_config.json")
for task in ["opportunity_type", "geographic_scope"]:
    d = f"{M}/classification/{task}"
    if trained(d): print(f"✔ {task}: مدرب مسبقاً في Drive — تخطٍ"); continue
    t = time.time()
    subprocess.run([sys.executable, "-m", "src.training.train_classifier", "--task", task], cwd=ROOT, env=os.environ.copy(), check=True)
    print(f"⏱ {task}: {(time.time()-t)/60:.1f} دقيقة")

In [ ]:
d = f"{M}/ner"
if trained(d): print("✔ NER: مدرب مسبقاً في Drive — تخطٍ")
else:
    t = time.time()
    subprocess.run([sys.executable, "-m", "src.training.train_ner"], cwd=ROOT, env=os.environ.copy(), check=True)
    print(f"⏱ NER: {(time.time()-t)/60:.1f} دقيقة")

## 3) التقييم المستقل على test + المقارنة بخطوط الأساس

In [ ]:
R = os.environ["LMI_REPORTS_DIR"]
for task in ["opportunity_type", "geographic_scope"]:
    if not os.path.exists(f"{R}/evaluation/classification_{task}/metrics.json"):
        !python -m src.evaluation.evaluate_classifier --task {task}
if not os.path.exists(f"{R}/evaluation/ner/metrics.json"):
    !python -m src.evaluation.evaluate_ner
!python scripts/build_results_section.py
shutil.copy(f"{ROOT}/docs/RESULTS.md", f"{PROJ}/docs/RESULTS.md")
from IPython.display import Markdown, display
display(Markdown(open(f"{PROJ}/docs/RESULTS.md", encoding="utf-8").read()))

## 4) الاختبارات

In [ ]:
!python -m pytest -q tests/test_dl_system.py -p no:cacheprovider 2>&1 | tail -3

## 5) تشغيل التطبيق مربوطاً بالنماذج
يقرأ البيانات من النسخة المحلية والنماذج والتقارير من Drive. يظهر رابط `trycloudflare.com` — افتحه. أوقف الخلية لإيقاف التطبيق.

In [ ]:
import re, subprocess, time
if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
app = subprocess.Popen(["streamlit", "run", "app_dl/main.py", "--server.port", "8501", "--server.headless", "true"],
                       cwd=ROOT, env=os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if m: print("🌐 رابط التطبيق:", m.group(0)); break
try:
    while True: time.sleep(60)
except KeyboardInterrupt:
    app.terminate(); tun.terminate(); print("أُوقف التطبيق")

## 6) (اختياري) تشغيل التطبيق على الكمبيوتر
ينشئ `lmi_models.zip` (أوزان fp16) و`lmi_reports.zip` في Drive (≈ نصف الحجم). نزّله من Drive وفكّه داخل مجلد المشروع على جهازك ثم:
```
pip install -r requirements.txt -r requirements_dl.txt
streamlit run app_dl/main.py
```

In [ ]:
!python scripts/package_results.py --out {MY} --fp16
!ls -la {MY}/lmi_models.zip {MY}/lmi_reports.zip